
Блокнот для обнаружения мошеннических транзакций.

Описание:
- Загружаем и предобрабатываем данные транзакций и идентификаторов
- Обучаем модели SVM и LogisticRegression на подмножестве данных для ускорения экспериментов
- Выводим метрики качества моделей (accuracy, precision, recall, F1)


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')  # Отключаем предупреждения

Настройка: ограничение на число строк для быстрой итерации

In [2]:
ROW_LIMIT = 10000

Функция загрузки данных

Параметр nrows позволяет читать только часть данных для ускорения загрузки и отладки


In [3]:
def load_data(nrows=None):
    print("Загрузка данных...")
    # Загрузка транзакций
    train_trans = pd.read_csv(
        'drive-download-20250512T040309Z-1-001/train_transaction.csv',
        nrows=nrows
    )
    test_trans = pd.read_csv(
        'drive-download-20250512T040309Z-1-001/test_transaction.csv',
        nrows=nrows
    )

    # Загрузка идентификаторов
    train_id = pd.read_csv(
        'drive-download-20250512T040309Z-1-001/train_identity.csv',
        nrows=nrows
    )
    test_id = pd.read_csv(
        'drive-download-20250512T040309Z-1-001/test_identity.csv',
        nrows=nrows
    )

    # Объединяем транзакции и идентификаторы по TransactionID
    train = pd.merge(train_trans, train_id, on='TransactionID', how='left')
    test = pd.merge(test_trans, test_id, on='TransactionID', how='left')
    return train, test


Функция предобработки данных

Заполняем пропуски, кодируем категории и масштабируем числовые признаки

In [4]:
def preprocess_data(train, test):
    print("Предобработка данных...")
    # Выделяем целевую переменную
    y = train['isFraud']
    train = train.drop(['isFraud', 'TransactionID'], axis=1)
    test  = test.drop(['TransactionID'], axis=1)


    # Оставляем только общие столбцы между train и test
    common_cols = list(set(train.columns) & set(test.columns))
    train = train[common_cols]
    test = test[common_cols]

    # Определяем категориальные и числовые колонки
    categorical_cols = train.select_dtypes(include=['object']).columns
    numerical_cols = train.select_dtypes(include=['int64', 'float64']).columns

    # Заполняем пропущенные значения
    for col in categorical_cols:
        train[col].fillna('missing', inplace=True)
        test[col].fillna('missing', inplace=True)
    for col in numerical_cols:
        median = train[col].median()
        train[col].fillna(median, inplace=True)
        test[col].fillna(median, inplace=True)

    # Кодируем категориальные признаки простым отображением
    for col in categorical_cols:
        unique_vals = pd.concat([train[col], test[col]]).unique()
        mapping = {val: idx for idx, val in enumerate(unique_vals)}
        train[col] = train[col].map(mapping)
        test[col] = test[col].map(mapping)

    # Масштабируем числовые признаки
    scaler = StandardScaler()
    train[numerical_cols] = scaler.fit_transform(train[numerical_cols])
    test[numerical_cols] = scaler.transform(test[numerical_cols])

    return train, test, y


Функция для вывода метрик качества модели

In [5]:
def show_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}\n") 


 Загрузка и предобработка подвыборки данных

In [6]:
train, test = load_data(nrows=ROW_LIMIT)
X, X_test_full, y = preprocess_data(train, test)

# Делим на обучающую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(
X, y, test_size=0.2, random_state=42, stratify=y
)

Загрузка данных...
Предобработка данных...


Обучение SVM

In [7]:
print("Обучение SVM...")
model = SVC()
model.fit(X_train, y_train)


Обучение SVM...


SVC()

In [8]:
print("Метрики SVM на обучении:")
y_pred_train = model.predict(X_train)
show_metrics(y_true=y_train, y_pred=y_pred_train)

Метрики SVM на обучении:
Accuracy:  0.9805
Precision: 1.0000
Recall:    0.2642
F1 Score:  0.4179



In [9]:
print("Метрики SVM на валидации:")
y_pred_val = model.predict(X_val)
show_metrics(y_true=y_val, y_pred=y_pred_val)

Метрики SVM на валидации:
Accuracy:  0.9770
Precision: 1.0000
Recall:    0.1321
F1 Score:  0.2333




Обучение Logistic Regression

In [10]:
print("Обучение Logistic Regression...")
model = LogisticRegression(penalty=None, solver='sag', max_iter=1000)
model.fit(X_train, y_train)

Обучение Logistic Regression...


LogisticRegression(max_iter=1000, penalty=None, solver='sag')

In [11]:
print("Метрики Logistic Regression на обучении:")
y_pred_train = model.predict(X_train)
show_metrics(y_true=y_train, y_pred=y_pred_train)

Метрики Logistic Regression на обучении:
Accuracy:  0.9831
Precision: 0.9326
Recall:    0.3915
F1 Score:  0.5515



In [12]:
print("Метрики Logistic Regression на валидации:")
y_pred_val = model.predict(X_val)
show_metrics(y_true=y_val, y_pred=y_pred_val)


Метрики Logistic Regression на валидации:
Accuracy:  0.9765
Precision: 0.6154
Recall:    0.3019
F1 Score:  0.4051

